In [0]:
%pip install geopy


In [0]:
dbutils.library.restartPython()

In [0]:
# Read from Unity Catalog using Spark
spark_df = spark.read.table("labuser_1_dev.default.earthquake_catalog_bronze")

# Convert to pandas DataFrame
df = spark_df.toPandas()
display(df)

In [0]:
import pandas as pd

df_earthquake_catalog = df.copy() 
 
df_earthquake_catalog['properties_time_dt'] = pd.to_datetime(df['properties_time'], unit='ms')\
    .dt.tz_localize('UTC')\
    .dt.tz_convert('Asia/Kuala_Lumpur')
df_earthquake_catalog['properties_updated_dt'] = pd.to_datetime(df['properties_updated'], unit='ms')\
    .dt.tz_localize('UTC')\
    .dt.tz_convert('Asia/Kuala_Lumpur')    

display(df_earthquake_catalog)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType , ArrayType , TimestampType

spark = SparkSession.builder.getOrCreate()

# Define schema
schema = StructType([
    StructField("type", StringType(), True),
    StructField("id", StringType(), True),

    StructField("properties_mag", DoubleType(), True),
    StructField("properties_place", StringType(), True),
    StructField("properties_time", LongType(), True),
    StructField("properties_updated", LongType(), True),
    StructField("properties_tz", StringType(), True),
    StructField("properties_url", StringType(), True),
    StructField("properties_detail", StringType(), True),
    StructField("properties_felt", DoubleType(), True),
    StructField("properties_cdi", DoubleType(), True),
    StructField("properties_mmi", DoubleType(), True),
    StructField("properties_alert", StringType(), True),
    StructField("properties_status", StringType(), True),
    StructField("properties_tsunami", IntegerType(), True),
    StructField("properties_sig", IntegerType(), True),
    StructField("properties_net", StringType(), True),
    StructField("properties_code", StringType(), True),
    StructField("properties_ids", StringType(), True),
    StructField("properties_sources", StringType(), True),
    StructField("properties_types", StringType(), True),
    StructField("properties_nst", LongType(), True),
    StructField("properties_dmin", DoubleType(), True),
    StructField("properties_rms", DoubleType(), True),
    StructField("properties_gap", LongType(), True),
    StructField("properties_magType", StringType(), True),
    StructField("properties_type", StringType(), True),
    StructField("properties_title", StringType(), True),

    StructField("geometry_type", StringType(), True),

    # If geometry_coordinates is GeoJSON point: [lon, lat, depth]
    StructField("geometry_coordinates", ArrayType(DoubleType(), containsNull=True), True),

    StructField("geometry_longitude", DoubleType(), True),
    StructField("geometry_latitude", DoubleType(), True),
    StructField("geometry_depth", DoubleType(), True),

    # Spark timestamps don’t store a timezone in the type; timezone handling is via session config.
    StructField("properties_time_dt", TimestampType(), True),
    StructField("properties_updated_dt", TimestampType(), True),
])

# Convert pandas to Spark
spark_df = spark.createDataFrame(df_earthquake_catalog, schema=schema)

table_name = "labuser_1_dev.default.earthquake_catalog_silver"

# Perform upsert
if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)
    
    (delta_table.alias("target")
        .merge(spark_df.alias("source"), "target.id = source.id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Merged data into {table_name}")
else:
    spark_df.write.format("delta").saveAsTable(table_name)
    print(f"Created new table {table_name}")

In [0]:
%skip
from geopy.geocoders import Nominatim
import time

# Initialize geolocator
geolocator = Nominatim(user_agent="earthquake_app")

def get_location_details(lat, lon):
    try:
        time.sleep(1)
        location = geolocator.reverse(f"{lat}, {lon}", language='en')
        addr = location.raw['address']
        
        return pd.Series({
            'country': str(addr.get('country', 'Unknown')),
            'country_code': str(addr.get('country_code', 'Unknown')),
            'state': str(addr.get('state', 'Unknown')),
            'city': str(addr.get('city') or addr.get('town') or addr.get('village', 'Unknown')),
            'full_address': str(location.address)
        })
    except:
        return pd.Series({
            'country': 'Unknown',
            'country_code': 'Unknown',
            'state': 'Unknown',
            'city': 'Unknown',
            'full_address': 'Unknown'
        })
     
df_earthquake_catalog[['geopy_country', 'geopy_country_code', 'geopy_state', 'geopy_city','geopy_full_address']] = df_earthquake_catalog.apply(
    lambda row: get_location_details(row['geometry_latitude'], row['geometry_longitude']),
    axis=1
)

# Ensure all new columns are strings (handle None/NaN values)
for col in ['geopy_country', 'geopy_country_code', 'geopy_state', 'geopy_city', 'geopy_full_address']:
    df_earthquake_catalog[col] = df_earthquake_catalog[col].fillna('Unknown').astype(str)

# Drop ALL problematic columns (add any other problematic column names here)
columns_to_drop = ['location', 'raw', 'address_obj']  # Add any other non-serializable columns
df_earthquake_catalog = df_earthquake_catalog.drop(columns=columns_to_drop, errors='ignore')
    
display(df_earthquake_catalog)